# SPY 0DTE ORB — Filter Sweep on the Top Family

The unfiltered ORB strategy collapsed over a full year (both top 22-day configs went to ~zero). This run tests whether adding filters can recover edge.

**Filters under test:**
1. **`min_break_pct`** — require the break to clear ORH/ORL by N% (filters weak/wick-only breaks)
2. **`vol_mult`** — require break-bar volume ≥ N × 20-bar avg (real moves have volume)
3. **`vwap`** — longs must close above VWAP; shorts below (filter chop)
4. **`pm-bias`** — longs only if premarket trended up; shorts only if down

Sweep is locked to config A (`or30 | conf=any | pt10 | sl10 | co11:30`) over the full year already cached. Plays through all 36 filter combinations (3 × 3 × 2 × 2) in ~30 seconds (cache is warm).

## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Filter sweep on the top config (year-long, cache warm)

Sweeps 36 filter combos × 1 base config × 1 year. Saves to `results/filtered_*.csv`.

In [ ]:
import shutil, os
os.makedirs('results', exist_ok=True)

!python -m iron_condor.cli --days 365 --sweep \
    --or-window 30 --confluence any --pt 0.10 --sl 0.10 --time-stop 30 --co 11:30 \
    --min-break 0 --min-break 0.0005 --min-break 0.001 \
    --vol-mult 0 --vol-mult 1.5 --vol-mult 2.0 \
    --vwap yes --vwap no \
    --pm-bias yes --pm-bias no

shutil.copy('results/sweep_trades.csv', 'results/filtered_trades.csv')
shutil.copy('results/sweep_summary.csv', 'results/filtered_summary.csv')
print('Done.')

## 4. Top 20 — does any filter combo make money?

In [ ]:
import pandas as pd
summary = pd.read_csv('results/filtered_summary.csv')
summary.head(20)

## 5. Per-filter isolation — which filter actually adds edge?

Group the 36 configs by each filter setting and show average return. Compares apples-to-apples since other dimensions are held constant.

In [ ]:
trades = pd.read_csv('results/filtered_trades.csv')
taken = trades[trades['exit_reason'].isin(['profit', 'stop', 'time'])]
if not taken.empty:
    for col in ['min_break_pct', 'vol_mult', 'vwap_filter', 'premarket_bias']:
        print(f'\n=== Average trade by {col} ===')
        grouped = taken.groupby(col).agg(
            trades=('net_pnl', 'count'),
            win_rate=('net_pnl', lambda s: (s > 0).mean()),
            avg_pnl=('net_pnl', 'mean'),
            median_pnl=('net_pnl', 'median'),
            total_pnl=('net_pnl', 'sum'),
        ).round(2)
        print(grouped)

## 6. Equity curves — top 5 configs

In [ ]:
import matplotlib.pyplot as plt

top5_configs = summary.head(5)['config'].tolist()
fig, ax = plt.subplots(figsize=(10, 5))
for cfg in top5_configs:
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=cfg, linewidth=1)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Account balance ($)')
ax.set_title('Top-5 filtered config equity curves')
ax.legend(fontsize=7, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()